# Smith v. United States — GPT-5.5 (3rd round, 2026-05)

Latest OpenAI flagship (`gpt-5.5`, released April 2026, became ChatGPT default 2026-05-05). Continues the methodology of `smith_chat_gpt.ipynb` / `smith_chat_gpt_2.ipynb` and uses the same prompts, parser, and 100-completion sampling so the new data lines up with the older runs in `analyze_responses.ipynb`.

**Temperature is intentionally not passed.** The OpenAI docs for `gpt-5.5` do not confirm `temperature` as a supported parameter (recent reasoning-capable flagships often reject it); to avoid breaking the 100-call run, we rely on the model's default sampling. If the user later confirms `temperature` is accepted, set it explicitly here.

API keys are loaded from `.env` at the repo root via `python-dotenv`.

## Setup
Load libraries and the `.env` file (which lives at the repo root, one level above `code/`).

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown
import pandas as pd

load_dotenv(os.path.join('..', '.env'))

## Model + client
The family alias is pinned in `MODEL`. The probe cell below captures the exact dated snapshot the API resolves it to, so the precise micro-version used for the run is preserved in the notebook output.

In [ ]:
# Latest GPT-5.5 flagship (OpenAI) as of 2026-05-06.
# Dated snapshot at time of writing: gpt-5.5-2026-04-23.
# We pin the family alias; the exact dated snapshot the API resolves to is
# captured in the probe cell below (response.model).
MODEL = 'gpt-5.5'

# OpenAI() picks up OPENAI_API_KEY from the environment.
client = OpenAI()

In [ ]:
def get_completion(prompt: str) -> str:
    '''Generate a GPT-5.5 chat completion for `prompt`.'''
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return response.choices[0].message.content

In [ ]:
# Probe call — captures the exact dated snapshot the API resolves the alias to.
probe = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Reply with one word: ok'}],
)
print(f'Requested model: {MODEL}')
print(f'Resolved model:  {probe.model}')
print(f'Probe content:   {probe.choices[0].message.content!r}')

## Inputs
Load the statute and the *Smith* case summary. The notebook is run from `code/`, so we reach `data/` via `os.path.join('..', 'data')`.

In [ ]:
DATAPATH = os.path.join('..', 'data')

statute = open(os.path.join(DATAPATH, '18USC924c1.txt'), 'r').read()
case_summary = open(os.path.join(DATAPATH, 'smith_case_summary.txt'), 'r').read()

## Prompt with visible analysis (`prompt`)
First variant: model is asked to *show* its reasoning inside `<legal_interpretation>` tags before giving the structured answer. Used here once for sanity / qualitative inspection — **not** used in the 100-call loop.

In [ ]:
prompt = f'''
You are asked to consider a legal scenario from the perspective of an ordinary person, not a legal expert. Your task is to read a statute and a case summary, then decide whether the actions described constitute 'use' of a gun as defined by the statute.

First, please read the following statute:

<statute>
{statute}
</statute>

Now, read the following case summary:

<case_summary>
{case_summary}
</case_summary>

Before providing your final answer, please analyze the situation in detail inside <legal_interpretation> tags. Consider the facts of the case and how they relate to the language of the statute. Remember, you're approaching this as an average person, not a legal professional.

In your analysis, include the following:
1. Quote the relevant parts of the statute that define 'use' of a gun.
2. List the key actions Mr. Smith took with the gun, according to the case summary. Number each action.
3. Explain how an ordinary person might interpret these actions in relation to the statute.
4. Identify any aspects of the case that might influence your interpretation.
5. Present arguments for and against considering Mr. Smith's actions as 'use' of the gun.
6. Form your opinion on whether a reasonable person would consider that Mr. Smith 'used' his gun in the sense that the language of the statute requires.

Based on your analysis, please provide your answer in the following format:

ANSWER: [Your answer (yes or no)]
REASONING: [Your reasoning, explained in simple terms that an average person would use]

Please provide your response now.
'''

In [ ]:
response = get_completion(prompt)

In [ ]:
display(Markdown(response))

## Constrained prompt (`new_prompt`)
Second variant: same instructions, but the analytical steps are silent. The response is constrained to `ANSWER: <yes/no>` followed by `REASONING: <text>` so the parser can split cleanly. **This is the prompt used for the 100-call loop.**

In [ ]:
new_prompt = f'''
You are asked to consider a legal scenario from the perspective of an ordinary person, not a legal expert. Your task is to read a statute and a case summary, then decide whether the actions described constitute 'use' of a gun as defined by the statute.

First, please read the following statute:

<statute>
{statute}
</statute>

Now, read the following case summary:

<case_summary>
{case_summary}
</case_summary>

Before providing your final answer, please analyze the situation in detail. Consider the facts of the case and how they relate to the language of the statute. Remember, you're approaching this as an average person, not a legal professional.

Specifically, consider the following (but DO NOT include them in your response):
1. The relevant parts of the statute that define 'use' of a gun.
2. The key actions Mr. Smith took with the gun, according to the case summary. Number each action.
3. How an ordinary person might interpret these actions in relation to the statute.
4. Any aspects of the case that might influence your interpretation.
5. Arguments for and against considering Mr. Smith's actions as 'use' of the gun.
6. Your opinion on whether a reasonable person would consider that Mr. Smith 'used' his gun in the sense that the language of the statute requires.

Based on your analysis, please provide your answer in the following format:

ANSWER: [Your answer (yes or no)]
REASONING: [Your reasoning, explained in simple terms that an average person would use]

Please provide your response now.
'''

In [ ]:
test_response = get_completion(new_prompt)
display(Markdown(test_response))

## Quick 3-iteration parser check
Make sure the `ANSWER:` / `REASONING:` parsing handles the model's output before committing to 100 paid calls.

In [ ]:
test_responses = []
test_answers = []
test_reasoning = []

for i in range(3):
    t_response = get_completion(new_prompt)
    test_answers.append(t_response[t_response.find('ANSWER:') + 7:t_response.find('REASONING:')].strip())
    test_reasoning.append(t_response[t_response.find('REASONING:') + 10:].strip())
    test_responses.append(t_response)

test_df = pd.DataFrame({'response': test_responses, 'answer': test_answers, 'reasoning': test_reasoning})
display(test_df)

## Generate 100 completions and save
The expensive cell. Hits the API 100 times with no rate-limit handling — expect minutes of wall time.

In [ ]:
answers = []
reasoning = []

for i in range(100):
    response = get_completion(new_prompt)
    answer_start = response.find('ANSWER:') + 7
    reasoning_start = response.find('REASONING:') + 10
    answer = response[answer_start:response.find('REASONING:')].strip()
    reason = response[reasoning_start:].strip()
    answers.append(answer)
    reasoning.append(reason)

responses_df = pd.DataFrame({'answer': answers, 'reasoning': reasoning})
display(responses_df.head())

## Inspect the answer distribution
If any answers come back wrapped in markdown (e.g. `Yes**\n\n**`), normalize them with a `.loc[...]` fix-up here — see the `smith_chat_gpt_2.ipynb` cell 13 pattern.

In [ ]:
print(responses_df['answer'].value_counts())

## Save
Write the cleaned DataFrame to `data/`. The CSV name encodes the exact model family for downstream `analyze_responses.ipynb` consumption.

In [ ]:
OUT_CSV = os.path.join(DATAPATH, 'smith_responses_gpt_5_5.csv')
responses_df.to_csv(OUT_CSV, index=False)
print(f'Wrote {len(responses_df)} rows to {OUT_CSV}')